# Counterdiabatic control of an ADMET feature-Ising encoding

Bind the same molecular count fields used by the curation and end-to-end
workflows, construct a transverse-field interpolation, and evaluate the
first-order nested-commutator adiabatic gauge potential (AGP). Independent
dense-matrix references check the algebra on the bounded offline fixture.

The fixture uses public SMILES with invented labels, not ADMET measurements.
This notebook verifies a Hamiltonian construction and an approximate AGP;
it does not establish predictive accuracy, quantum advantage, or exact
counterdiabatic transport. See the README for setup and verification commands.

The operators encode information from molecular features in chosen dimensionless
energy units (ℏ = 1). They are not an electronic-structure Hamiltonian or a
calibrated physical model of an ADMET mechanism.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "admet_curation.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from the ADMET project or its pipeline directory")
sys.path[:0] = [str(PROJECT_ROOT / "src"), str(PROJECT_ROOT / "python_classes")]

from admet_curation import CurationConfig, FIXTURE_NAME, run_complete_curation, molecule_problem_input
from classical_preprocessing import (
    AGPSettings, build_problem_hamiltonian, build_initial_hamiltonian,
    recalculate_counterdiabatic_hamiltonian,
)

MODE = "fixture"
DATASET = FIXTURE_NAME
SAMPLE_INDEX = 13
CONFIG = CurationConfig(top_k=4)
SETTINGS = AGPSettings(interpolation=0.5, lambda_dot=1.0)
curation = run_complete_curation(mode=MODE, config=CONFIG)
curated_adapter, feature_vector, provenance = molecule_problem_input(curation, DATASET, SAMPLE_INDEX)
display(provenance)

## 1. Molecular problem and driver

The interaction landscape contains every screened pair and triad. Molecular
fields are the selected molecule's scaled Morgan counts, rather than its
dataset's feature–label MI scores:

$$H_p(\theta)=\sum_i\theta_i Z_i+\sum_{i<j}J_{ij}Z_iZ_j
+\sum_{i<j<k}K_{ijk}Z_iZ_jZ_k,\qquad H_0=-\sum_iX_i.$$

With $\hbar=1$, use the dimensionless interpolation
$H_{\mathrm{ad}}(\lambda)=(1-\lambda)H_0+\lambda H_p$ and
$O_0=\partial_\lambda H_{\mathrm{ad}}=H_p-H_0$. Here $\dot\lambda$
is an explicit local schedule derivative; no full schedule is integrated.

In [ ]:
problem = build_problem_hamiltonian(curated_adapter, DATASET, feature_vector, None)
initial = build_initial_hamiltonian(len(feature_vector))
display(pd.DataFrame(initial))
display(pd.DataFrame(problem["pauli_terms"]))

## 2. Nested commutators and the variational coefficient

Form $O_1=[H_{\mathrm{ad}},O_0]$ and
$O_2=[H_{\mathrm{ad}},O_1]$. The one-commutator ansatz is

$$A^{(1)}_\lambda=i\alpha_1O_1,\qquad
\alpha_1=-\frac{\lVert O_1\rVert_{\mathrm{HS}}^2}
{\lVert O_2\rVert_{\mathrm{HS}}^2}.$$

Unique Pauli words are combined before computing the Hilbert–Schmidt norms;
the common $2^n$ factor cancels. $O_1$ is anti-Hermitian, while $O_0$, $O_2$,
and $A^{(1)}_\lambda$ are Hermitian. The coefficient minimizes the
Hilbert–Schmidt residual within this single-parameter ansatz. It is not an
exact AGP in general. A commuting path has the zero gauge potential.

In [ ]:
diagnostic_payload = recalculate_counterdiabatic_hamiltonian(
    curated_adapter, DATASET, feature_vector, SETTINGS, include_components=True)
components = diagnostic_payload.pop("agp_components")
display(pd.DataFrame([
    {"operator": name, "terms": len(terms),
     "Pauli_coefficient_norm_sq": sum(abs(complex(t["coefficient"]))**2 for t in terms)}
    for name, terms in components.items()
]))
display(diagnostic_payload["agp_metadata"])
display(pd.DataFrame(components["o1"]))
display(pd.DataFrame(components["o2"]))

## 3. Counterdiabatic snapshot and independent matrix verification

$$H_{\mathrm{CD}}=H_{\mathrm{ad}}+\dot\lambda A^{(1)}_\lambda.$$

For at most six system qubits, construct matrices independently from explicit
Pauli tensor products (qubit 0 is the leftmost factor). Compare the nested
commutators, variational coefficient, Hermiticity, and CD sum. Check that the
residual $G=O_0+\alpha_1O_2$ is stationary and no larger than the zero-AGP
residual. These checks validate the finite ansatz; they do not bound its
distance from the exact many-body AGP.

In [ ]:
from functools import reduce

PAULIS = {
    "I": np.eye(2, dtype=complex), "X": np.array([[0, 1], [1, 0]], complex),
    "Y": np.array([[0, -1j], [1j, 0]], complex), "Z": np.diag([1., -1.]).astype(complex),
}

def dense_reference(terms, n_qubits):
    result = np.zeros((2**n_qubits, 2**n_qubits), dtype=complex)
    for term in terms:
        operators = {int(token[1:]): token[0] for token in term["pauli_string"].split()}
        factors = [PAULIS[operators.get(q, "I")] for q in range(n_qubits)]
        result += complex(term["coefficient"]) * reduce(np.kron, factors)
    return result

n_qubits = len(feature_vector)
if n_qubits <= 6:
    matrices = {name: dense_reference(terms, n_qubits) for name, terms in components.items()}
    H = matrices["adiabatic"]
    O0 = matrices["problem"] - matrices["initial"]
    O1 = H @ O0 - O0 @ H
    O2 = H @ O1 - O1 @ H
    alpha = -float(np.vdot(O1, O1).real / np.vdot(O2, O2).real)
    A = 1j * alpha * O1
    CD = dense_reference(diagnostic_payload["pauli_terms"], n_qubits)
    G = O0 + alpha * O2
    checks = {
        "O0_error": np.linalg.norm(matrices["o0"]-O0),
        "O1_error": np.linalg.norm(matrices["o1"]-O1),
        "O2_error": np.linalg.norm(matrices["o2"]-O2),
        "alpha_error": abs(diagnostic_payload["agp_metadata"]["alpha1"]-alpha),
        "AGP_error": np.linalg.norm(matrices["agp"]-A),
        "CD_sum_error": np.linalg.norm(CD-(H+SETTINGS.lambda_dot*A)),
        "CD_Hermiticity_error": np.linalg.norm(CD-CD.conj().T),
        "residual_stationarity": abs(np.vdot(O2, G)),
    }
    assert max(checks.values()) < 1e-8, checks
    assert np.linalg.norm(G) <= np.linalg.norm(O0) + 1e-10
    display(pd.DataFrame([checks]))
    print({"zero_AGP_residual_HS": float(np.linalg.norm(O0)),
           "first_order_residual_HS": float(np.linalg.norm(G))})
else:
    print("Dense references skipped: exceeds the six-qubit verification budget")

## 4. Resource report and runtime handoff

The report counts classical Pauli terms, not compiled native gates or runtime
backend resources. No interaction-term truncation is applied. The optional
JSON snapshot contains finite real CD coefficients, molecular provenance, and
the configuration; diagnostic complex commutators remain local.

The end-to-end notebook performs LCU/QSVT preparation and offline handoff
validation. Evolution under this **frozen** $H_{\mathrm{CD}}(\lambda)$ is a
snapshot experiment. Demonstrating continuous counterdiabatic transport would
require a defined $\lambda(t)$, schedule integration, and transport-error
comparisons. API execution and backend validation are outside this notebook.

In [ ]:
display(pd.DataFrame([{
    "system_qubits": n_qubits,
    "retained_pair_terms": provenance["retained_pairs"],
    "retained_triad_terms": provenance["retained_triads"],
    "problem_terms": len(problem["pauli_terms"]),
    "CD_terms": len(diagnostic_payload["pauli_terms"]),
}]))

WRITE_ARTIFACTS = False
if WRITE_ARTIFACTS:
    from dataclasses import asdict
    from runtime_input_protocol import atomic_write_json
    artifact = {"schema_version": 2, "mode": MODE, "fixture_labels_are_synthetic": MODE == "fixture",
                "config": asdict(CONFIG), "provenance": provenance,
                "counterdiabatic_hamiltonian": {k: v for k, v in diagnostic_payload.items() if k != "agp_components"}}
    destination = PROJECT_ROOT / "artifacts" / "molecular_counterdiabatic.json"
    destination.parent.mkdir(parents=True, exist_ok=True)
    atomic_write_json(destination, artifact)
    print(destination)

## 5. Time-dependent transport and classical control references

Use the smooth schedule $s(u)=3u^2-2u^3$, with $u=t/T$ and vanishing endpoint
derivatives. Integrate

$$i\frac{d|\psi\rangle}{du}=\left[T H(s(u))+s'(u)A_s\right]|\psi\rangle$$

from the driver's unique ground state $|+\rangle^{\otimes n}$. Compare bare
annealing, the production first-order Pauli AGP, and the exact **ground-projector**
reference $A_g=i[\partial_sP_g,P_g]$. The dense reference handles excited-state
degeneracy without dividing by excited–excited gaps; an unresolved ground gap
fails explicitly. It is a state-specific classical reference, not an efficient
full-spectrum quantum algorithm.

Report ground fidelity, excitation energy, norm defect, sampled gap, control
cost, and variational residual. Do not assume first-order CD improves fidelity
at every duration. Independently refine a midpoint-exponential propagator at
the representative duration nearest $T=1$; compare all three methods with the
adaptive ODE solution. Verify three frozen CD QSVT snapshots against direct
matrix exponentiation. These operator snapshots do not implement time-ordered
QSVT annealing.

The initial ground state here differs from the feature RY state at the API
boundary. Both experiments are specified explicitly. The gap minimum is sampled,
not a certified lower bound over the continuous path. Times and energies use the
chosen dimensionless scale with ℏ = 1.


In [ ]:
from transport_benchmark import run_transport_benchmark
transport = run_transport_benchmark(curated_adapter, DATASET, feature_vector)
columns = ['duration', 'method', 'ground_fidelity', 'excitation_energy',
           'max_norm_error', 'sampled_peak_control_norm',
           'integrated_control_hs_squared_quadrature']
display(pd.DataFrame(transport['rows'])[columns])
print({'minimum_observed_gap': transport['observed_minimum_ground_gap'],
       'ground_reference': transport['exact_reference'],
       'problem_sha256': transport['problem_hamiltonian_sha256']})
display(pd.DataFrame(transport['convergence']))
display(pd.DataFrame([{'s': row['interpolation'],
    'operator_error': row['reference']['recovered_evolution_error'],
    'error_bound': row['reference']['certified_recovered_evolution_error_bound'],
    'success_probability_lower_bound': row['success_probability_lower_bound']}
    for row in transport['qsvt_snapshots']]))


## Interpretation and reproducible results

The benchmark is an ablation of a first-order variational control on one bounded
encoding. Its exact reference is a numerical calibration; its higher control
cost must remain visible. Repeated durations are not independent statistical
replicates. The complete source-hashed experiment, plots, and the separate real
BBB data sensitivity study are reproduced by `python run_benchmarks.py`; see
the README for the pinned public-data acquisition command and measured results.
